# 🎯 [DAT06_M04] 데이터 이상치(전월 대비 급증 항목) 자동 감지 및 이상 원인 요약 추천 모듈 고도화
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m04_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M04 |
| **미션 주제** | 데이터 이상치(전월 대비 급증 항목) 자동 감지 및 이상 원인 요약 추천 모듈 고도화 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 데이터 이상치(전월 대비 급증 항목) 자동 감지 및 이상 원인 요약 추천 모듈 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전월 대비 급증 항목 이상치 감지 및 이상 원인 요약 추천 모듈을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **전월 대비 급증한 품목을 자동 감지하고 이상 원인을 요약해 추천하는 모듈을 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **품목별 증가율의 로그 값에 중앙값과 MAD 기반 점수를 적용해 급증 품목을 찾는다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ai/dat06_intelligence_module.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추천 사유의 합리성 및 실무 수용성

#### 미션 PASS 조건

- 정량 기준: 추천/분류 정확도 85% 이상 달성


- 진행 기준: AI 추론 모듈 구현 및 벤치마크 테스트 완료


- 검증 방법: 과거 승인/처리 완료 데이터와의 추천 일치도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""전월 대비 급증한 품목을 로버스트 통계로 자동 감지하고, 훈련·정비 일정 등 맥락 정보와 대조해 이상 원인을 요약·추천하는 모듈을 구현해 감지 정확도와 원인 분류 정확도를 측정한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

# 1) 가상 2개월 데이터 + 정답(숨은 원인) 생성
led = pd.DataFrame(make_ledger(2))
p7 = led[led["month"] == "2026-07"].set_index(["unit", "item"])
p8 = led[led["month"] == "2026-08"].set_index(["unit", "item"])
d = pd.DataFrame({"cat": p8["cat"], "prev": p7["issue"].clip(lower=1), "cur": p8["issue"]})
rng = np.random.default_rng(4)
n = len(d)
d["train_days"] = rng.integers(0, 5, n)          # 해당 부대 월 훈련 일수(맥락)
d["repair_orders"] = rng.integers(0, 4, n)       # 정비 지시서 건수(맥락)
d["special"] = 0                                 # 일회성 특별 지급 승인 여부
d["truth"] = "정상"
d["cur"] = (d["prev"] * rng.uniform(0.7, 1.4, n)).round().astype(int)    # 정상 변동(-30%~+40%)
CAUSES = ["훈련 증가", "정비 집중", "입력 자릿수 오류", "일회성 특별 지급", "원인 불명"]
idx = rng.choice(n, 36, replace=False)
for k, i in enumerate(idx):
    c = CAUSES[k % 5]
    d.iloc[i, d.columns.get_loc("truth")] = c
    f = {"훈련 증가": rng.uniform(1.8, 3.0), "정비 집중": rng.uniform(1.8, 3.2), "입력 자릿수 오류": 10.0,
         "일회성 특별 지급": rng.uniform(2.0, 4.0), "원인 불명": rng.uniform(1.9, 3.0)}[c]
    d.iloc[i, d.columns.get_loc("cur")] = int(d.iloc[i]["prev"] * f)
    if c == "훈련 증가":
        d.iloc[i, d.columns.get_loc("train_days")] = rng.integers(8, 14)
    if c == "정비 집중":
        d.iloc[i, d.columns.get_loc("repair_orders")] = rng.integers(7, 12)
    if c == "일회성 특별 지급":
        d.iloc[i, d.columns.get_loc("special")] = 1
for i in rng.choice(n, 8, replace=False):        # 정상이지만 활동이 많았던 곳(오탐 유도)
    d.iloc[i, d.columns.get_loc("train_days")] = rng.integers(8, 14)

# 2) 감지: 품목별 로그 증가율의 중앙값·MAD 기반 z 점수
d["ratio"] = d["cur"] / d["prev"]
lr = np.log(d["ratio"])
med = lr.groupby(level="item").transform("median")
mad = (lr - med).abs().groupby(level="item").transform("median").clip(lower=0.05) * 1.4826
d["z"] = (lr - med) / mad
d["flag"] = (d["z"] > 2.0) & (d["ratio"] >= 1.6)

# 3) 원인 요약 추천(규칙 기반, 사람이 확인할 후보를 제시)
def recommend(r):
    if abs(r["ratio"] - 10) < 1.2:
        return "입력 자릿수 오류"
    if r["special"]:
        return "일회성 특별 지급"
    if r["repair_orders"] >= 6:
        return "정비 집중"
    if r["train_days"] >= 8:
        return "훈련 증가"
    return "원인 불명"
ACTION = {"훈련 증가": "훈련 계획 대조 후 정상 처리", "정비 집중": "정비 지시서와 소모 수량 대조", "입력 자릿수 오류": "제출 부대에 수치 재확인 요청",
          "일회성 특별 지급": "승인 문서 번호 확인", "원인 불명": "담당자가 부대에 직접 문의"}
d["cause"] = np.where(d["flag"], d.apply(recommend, axis=1), "정상")

# 4) 평가
truth_anom = d["truth"] != "정상"
tp = int((d["flag"] & truth_anom).sum()); fp = int((d["flag"] & ~truth_anom).sum()); fn = int((~d["flag"] & truth_anom).sum())
prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
acc = float((d["flag"] == truth_anom).mean())
both = d[d["flag"] & truth_anom]
cause_acc = float((both["cause"] == both["truth"]).mean())
total_acc = float(((d["cause"] == d["truth"])).mean())
print(f"[이상 감지] 대상 {n}건(부대 24 x 품목 11), 실제 급증 {int(truth_anom.sum())}건 [교육용가상자료]")
print(f"감지 {int(d['flag'].sum())}건: 정확도 {acc:.1%}, 정밀도 {prec:.1%}, 재현율 {rec:.1%} (TP {tp}, FP {fp}, FN {fn})")
print(f"원인 분류 정확도(감지된 실제 급증 중) {cause_acc:.1%}, 전체 분류(정상 포함) 정확도 {total_acc:.1%}")
print("\n[원인별 감지·추천 현황]")
print(pd.crosstab(both["truth"], both["cause"]).to_string())
print("\n[이상 원인 요약 추천 예시 - 증가율 상위 5건]")
for (u, it), r in d[d["flag"]].sort_values("ratio", ascending=False).head(5).iterrows():
    print(f" - {u} {it}: 전월 {int(r['prev'])} -> 당월 {int(r['cur'])}({r['ratio']:.1f}배), 추정 원인 '{r['cause']}' -> 조치: {ACTION[r['cause']]}")
print("추천은 후보 제시이며 최종 원인 판단과 수치 정정은 담당자가 확인한다.")
print(f"\n목표(추천·분류 정확도 85% 이상): 감지 {'달성' if acc >= 0.85 else '미달'}, 원인 분류 {'달성' if cause_acc >= 0.85 else '미달'}")


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[이상 감지] 대상 264건(부대 24 x 품목 11), 실제 급증 36건 [교육용가상자료]
감지 35건: 정확도 99.6%, 정밀도 100.0%, 재현율 97.2% (TP 35, FP 0, FN 1)
원인 분류 정확도(감지된 실제 급증 중) 100.0%, 전체 분류(정상 포함) 정확도 99.6%

[원인별 감지·추천 현황]
cause      원인 불명  일회성 특별 지급  입력 자릿수 오류  정비 집중  훈련 증가
truth                                               
원인 불명          6          0          0      0      0
일회성 특별 지급      0          7          0      0      0
입력 자릿수 오류      0          0          7      0      0
정비 집중          0          0          0      7      0
훈련 증가          0          0          0      0      8

[이상 원인 요약 추천 예시 - 증가율 상위 5건]
 - 가상01대대 훈련탄B: 전월 93 -> 당월 930(10.0배), 추정 원인 '입력 자릿수 오류' -> 조치: 제출 부대에 수치 재확인 요청
 - 가상15대대 피복류: 전월 119 -> 당월 1190(10.0배), 추정 원인 '입력 자릿수 오류' -> 조치: 제출 부대에 수치 재확인 요청
 - 가상15대대 정비소모품: 전월 124 -> 당월 1240(10.0배), 추정 원인 '입력 자릿수 오류' -> 조치: 제출 부대에 수치 재확인 요청
 - 가상19대대 모의탄: 전월 174 -> 당월 1740(10.0배), 추정 원인 '입력 자릿수 오류' -> 조치: 제출 부대에 수치 재확인 요청
 - 가상21대대 정비소모품: 전월 146 -> 당월 1460(10.0배), 추정 원인 '입력 자릿수 오류' -> 조치: 제출 부대에 수치 재확인 요청
추천은 후보 제시이며 최종 원인 판단과 수치 정정은 담당자가 확인한다.

목표(추천·분류 정확도 85% 이상): 감지 달성, 원인 분류 달성
```
</details>
